# Prompt Block 1 — Data Ingestion, Crop Standardisation & Feature Enrichment

**Explainable AI-Based Crop Yield Prediction and Farm Decision Support System**

This notebook block:
1. Imports all required libraries.
2. Loads the four raw CSVs with their **actual column names** and standardises crop names.
3. Computes per-crop aggregate tables from `smart_farming.csv` and `crop_recommendation.csv`.
   - **Note:** `soil_weather.csv` has no `Crop` column, so per-crop aggregation is not possible from it.
4. Left-joins those aggregates onto the backbone `crop_yield.csv`.
5. Adds Gaussian jitter to the newly joined columns (augmentation, not ground truth).
6. Saves the result as `agriculture_data.csv`.
7. Prints a comprehensive summary.

In [ ]:
# ====================================================================
# 1.  IMPORTS
# ====================================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from io import StringIO

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & modelling
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    explained_variance_score, mean_absolute_percentage_error
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

# Boosting
import xgboost as xgb
import lightgbm as lgb

# Explainability
import shap

# Categorical encoding
import category_encoders as ce

# Model persistence
import joblib

print("All libraries imported successfully.")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")
print(f"xgboost {xgb.__version__}  |  lightgbm {lgb.__version__}")
print(f"scikit-learn  |  shap  |  category_encoders  |  joblib  OK")

---
## 2.  Load the Four Raw CSVs (Real Data)

In [ ]:
# ====================================================================
# 2.  LOAD RAW CSVs
# ====================================================================

BASE = "../data"

# --- Backbone: crop_yield.csv ---
crop_yield = pd.read_csv(f"{BASE}/crop_yield.csv")
print("=== crop_yield.csv (backbone) ===")
print(f"Shape: {crop_yield.shape}")
print(f"Columns: {list(crop_yield.columns)}")
print(f"Unique crops: {sorted(crop_yield['Crop'].unique())}")
print()

# --- soil_weather.csv ---
# NOTE: This file has NO 'Crop' column, so per-crop aggregation is impossible.
soil_weather = pd.read_csv(f"{BASE}/soil_weather.csv")
print("=== soil_weather.csv (no Crop column) ===")
print(f"Shape: {soil_weather.shape}")
print(f"Columns: {list(soil_weather.columns)}")
print()

# --- smart_farming.csv ---
# Crop column is 'crop_type'. Has: soil_pH, humidity_%, sunlight_hours, irrigation_type, etc.
smart_farming = pd.read_csv(f"{BASE}/smart_farming.csv")
print("=== smart_farming.csv ===")
print(f"Shape: {smart_farming.shape}")
print(f"Columns: {list(smart_farming.columns)}")
print(f"Unique crop_type: {sorted(smart_farming['crop_type'].unique())}")
print()

# --- crop_recommendation.csv ---
# Crop column is 'label'. Has: N, P, K, temperature, humidity, ph, rainfall
crop_recommendation = pd.read_csv(f"{BASE}/crop_recommendation.csv")
print("=== crop_recommendation.csv ===")
print(f"Shape: {crop_recommendation.shape}")
print(f"Columns: {list(crop_recommendation.columns)}")
print(f"Unique labels: {sorted(crop_recommendation['label'].unique())}")

---
## 3.  Standardise the Crop Column Across All Dataframes

- `smart_farming.csv` uses `crop_type` -> renamed to `Crop`
- `crop_recommendation.csv` uses `label` -> renamed to `Crop`
- `soil_weather.csv` has **no crop column** 

Crop names are lowercased, stripped, and known synonyms mapped (one-directional).

In [ ]:
# ====================================================================
# 3.  CROP NAME STANDARDISATION
# ====================================================================

def standardise_crop_names(series: pd.Series) -> pd.Series:
    """Lowercase, strip whitespace, and map common synonyms to a canonical form."""
    synonym_map = {
        "maize":      "corn",
        "sweet corn": "corn",
        "baby corn":  "corn",
    }
    s = series.str.strip().str.lower()
    return s.map(lambda x: synonym_map.get(x, x))


# Rename columns to standard 'Crop'
smart_farming = smart_farming.rename(columns={"crop_type": "Crop"})
crop_recommendation = crop_recommendation.rename(columns={"label": "Crop"})

# Track before counts
before_counts = {}
for name, df in [("crop_yield", crop_yield), ("smart_farming", smart_farming), ("crop_recommendation", crop_recommendation)]:
    if "Crop" in df.columns:
        before_counts[name] = df["Crop"].nunique()

# Standardise
crop_yield["Crop"] = standardise_crop_names(crop_yield["Crop"])
smart_farming["Crop"] = standardise_crop_names(smart_farming["Crop"])
crop_recommendation["Crop"] = standardise_crop_names(crop_recommendation["Crop"])

print("Crop standardisation complete.")
for name, df in [("crop_yield", crop_yield), ("smart_farming", smart_farming), ("crop_recommendation", crop_recommendation)]:
    after = df["Crop"].nunique()
    print(f"  {name:25s}: {before_counts[name]:>3d} unique -> {after:>3d} unique  |  crops: {sorted(df['Crop'].unique())}")
print(f"\n  soil_weather:            No 'Crop' column -- cannot compute per-crop aggregates.")

---
## 4.  Compute Per-Crop Aggregate Tables

### Files with a Crop column:
- **smart_farming.csv** -> per-crop mean of `soil_pH`, `humidity_%`, `sunlight_hours`; mode of `irrigation_type`
- **crop_recommendation.csv** -> per-crop mean of `N`, `P`, `K`, `temperature`, `humidity`, `ph`, ` rainfall`

### soil_weather.csv
No `Crop` column -- only global means can be computed.

In [ ]:
# ====================================================================
# 4a.  soil_weather -> global means (no Crop column)
# ====================================================================

print("Global means from soil_weather.csv (per-crop grouping impossible):")
for col in soil_weather.columns:
    val = soil_weather[col].mean()
    print(f"  {col:15s} = {val:.4f}")

In [ ]:
# ====================================================================
# 4b.  smart_farming -> per-crop aggregates
# ====================================================================

agg_smart_farming = (
    smart_farming
    .groupby("Crop", as_index=False)
    .agg(
        soil_pH=("soil_pH", "mean"),
        humidity_pct=("humidity_%", "mean"),
        sunlight_hours=("sunlight_hours", "mean"),
        irrigation_type=("irrigation_type", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    )
)

print(f"Per-crop aggregate from smart_farming: {agg_smart_farming.shape[0]} crops")
display(agg_smart_farming)
print(f"\nBackbone crops: {sorted(crop_yield['Crop'].unique())}")
backbone_not_in_sf = set(crop_yield['Crop'].unique()) - set(agg_smart_farming['Crop'].unique())
if backbone_not_in_sf:
    print(f"Crops in backbone but NOT in smart_farming: {backbone_not_in_sf} -- will get global fallback.")

In [ ]:
# ====================================================================
# 4c.  crop_recommendation -> per-crop aggregates
# ====================================================================

agg_crop_rec = (
    crop_recommendation
    .groupby("Crop", as_index=False)
    .agg(
        N=("N", "mean"),
        P=("P", "mean"),
        K=("K", "mean"),
        temperature=("temperature", "mean"),
        humidity=("humidity", "mean"),
        ph=("ph", "mean"),
        rainfall=("rainfall", "mean"),
    )
)

print(f"Per-crop aggregate from crop_recommendation: {agg_crop_rec.shape[0]} crops")
display(agg_crop_rec.head())
backbone_in_cr = set(crop_yield['Crop'].unique()) & set(agg_crop_rec['Crop'].unique())
backbone_not_in_cr = set(crop_yield['Crop'].unique()) - set(agg_crop_rec['Crop'].unique())
print(f"\nBackbone crops found in crop_recommendation: {sorted(backbone_in_cr)}")
if backbone_not_in_cr:
    print(f"Backbone crops NOT found in crop_recommendation: {backbone_not_in_cr} -- will get global fallback.")

---
## 5.  Left-Join Aggregates onto `crop_yield`

Rows in `crop_yield` whose crop is not in a supplementary dataset get `NaN`, then global fallback.

In [ ]:
# ====================================================================
# 5.  LEFT-JOIN AGGREGATES
# ====================================================================

enriched = crop_yield.copy()
n_before = len(enriched)
join_stats = {}

# --- Join smart_farming aggregate ---
enriched = enriched.merge(agg_smart_farming, on="Crop", how="left", suffixes=("", "_sf"))
matched_sf = enriched["soil_pH"].notna().sum()
join_stats["smart_farming"] = {"matched": matched_sf, "fallback": n_before - matched_sf}

# --- Join crop_recommendation aggregate ---
enriched = enriched.merge(agg_crop_rec, on="Crop", how="left", suffixes=("", "_cr"))
matched_cr = enriched["N"].notna().sum()
join_stats["crop_recommendation"] = {"matched": matched_cr, "fallback": n_before - matched_cr}

print(f"Joined enriched dataframe: {enriched.shape}")
for src, st in join_stats.items():
    pct = st["matched"] / n_before * 100
    print(f"  {src:25s}: {st['matched']:>10,} matched ({pct:.2f}%)  |  {st['fallback']:>8,} fallback")
print(f"  {'soil_weather':25s}: {'N/A':>10}  (no Crop column to join on)")

### 5b.  Global Mean Fallback for Unmatched Crops

In [ ]:
# Compute global fallback values
irrigation_mode = smart_farming["irrigation_type"].mode()
irrigation_fallback = irrigation_mode.iloc[0] if not irrigation_mode.empty else "Unknown"

global_fallback = {
    "soil_pH": smart_farming["soil_pH"].mean(),
    "humidity_pct": smart_farming["humidity_%"].mean(),
    "sunlight_hours": smart_farming["sunlight_hours"].mean(),
    "irrigation_type": irrigation_fallback,
    "N": crop_recommendation["N"].mean(),
    "P": crop_recommendation["P"].mean(),
    "K": crop_recommendation["K"].mean(),
    "temperature": crop_recommendation["temperature"].mean(),
    "humidity": crop_recommendation["humidity"].mean(),
    "ph": crop_recommendation["ph"].mean(),
    "rainfall": crop_recommendation["rainfall"].mean(),
}

# Fill NaN with global values
fallback_counts = {}
for col, fb_val in global_fallback.items():
    if col in enriched.columns:
        n_nan = enriched[col].isna().sum()
        if n_nan > 0:
            fallback_counts[col] = n_nan
            enriched[col] = enriched[col].fillna(fb_val)

print("Fallback to global means/mode applied:")
if fallback_counts:
    for col, cnt in fallback_counts.items():
        val = global_fallback[col]
        if isinstance(val, (int, float)):
            print(f"  {col:20s}: {cnt:>8,} rows filled with global mean ({val:.4f})")
        else:
            print(f"  {col:20s}: {cnt:>8,} rows filled with global mode ({val})")
else:
    print("  (No columns needed fallback -- all crops matched.)")

---
## 6.  Gaussian Jitter (+/-5%) to Newly Joined Numeric Columns

**Rationale:** Since aggregates are per-crop means, rows sharing a crop get identical values. Gaussian noise (+/-5% relative) preserves the signal while adding realistic variation.

**Important:** Synthetic augmentation -- not row-level ground truth.

In [ ]:
# ====================================================================
# 6.  GAUSSIAN JITTER
# ====================================================================

RNG = np.random.default_rng(seed=42)

jitter_cols = ["soil_pH", "humidity_pct", "sunlight_hours", "N", "P", "K", "temperature", "humidity", "ph", "rainfall"]

print("Applying Gaussian jitter (+/-5% relative):")
for col in jitter_cols:
    if col in enriched.columns and np.issubdtype(enriched[col].dtype, np.number):
        col_mean = enriched[col].mean()
        noise_std = 0.05 * col_mean
        noise = RNG.normal(loc=0, scale=noise_std, size=len(enriched))
        enriched[col] = enriched[col] + noise
        print(f"  {col:20s}  mean={col_mean:.4f}  noise_std={noise_std:.4f}")

print("\nJitter complete.")

---
## 7.  Save Master Dataset as `agriculture_data.csv`

In [ ]:
# ====================================================================
# 7.  SAVE MASTER DATASET
# ====================================================================

output_path = f"{BASE}/agriculture_data.csv"
enriched.to_csv(output_path, index=False)
print(f"Master dataset saved to {output_path}")
print(f"Final shape: {enriched.shape[0]:,} rows x {enriched.shape[1]} columns")

---
## 8.  Final Summary

In [ ]:
# ====================================================================
# 8a.  SHAPE & INFO
# ====================================================================

print("=" * 72)
print("  FINAL SUMMARY -- agriculture_data.csv")
print("=" * 72)
print(f"\nShape: {enriched.shape[0]:,} rows x {enriched.shape[1]} columns")

buf = StringIO()
enriched.info(buf=buf)
print(buf.getvalue())

In [ ]:
# ====================================================================
# 8b.  DESCRIPTIVE STATISTICS
# ====================================================================

print("Descriptive statistics (numeric columns):")
display(enriched.describe().round(3))

print("\nDescriptive statistics (object/categorical columns):")
cat_cols = enriched.select_dtypes(include=["object", "bool"]).columns
display(enriched[cat_cols].describe())

In [ ]:
# ====================================================================
# 8c.  DTYPES & MISSING VALUES
# ====================================================================

print("Column dtypes:")
for dtype, count in enriched.dtypes.value_counts().items():
    print(f"  {str(dtype):15s} : {count} columns")

print()
missing = enriched.isna().sum()
missing_pct = enriched.isna().mean() * 100
missing_df = pd.DataFrame({"Missing_Count": missing, "Missing_Pct": missing_pct})
missing_df = missing_df[missing_df["Missing_Count"] > 0]

if len(missing_df) > 0:
    print("Columns with missing values:")
    display(missing_df.round(4))
else:
    print("No missing values in the dataset -- all columns are complete.")

In [ ]:
# ====================================================================
# 8d.  DUPLICATES & JOIN STATISTICS
# ====================================================================

n_duplicates = enriched.duplicated().sum()
print(f"Duplicate rows: {n_duplicates:,}  ({n_duplicates / len(enriched) * 100:.4f}%)")

key_cols = ["Crop", "Region", "Soil_Type"]
n_dup_key = enriched.duplicated(subset=key_cols).sum()
print(f"Duplicate (Crop + Region + Soil_Type): {n_dup_key:,}")

print()
print("Join statistics:")
print(f"  Backbone rows: {n_before:,}")
for src, st in join_stats.items():
    pct = st["matched"] / n_before * 100
    print(f"  {src:25s}: {st['matched']:>10,} matched ({pct:.2f}%)  |  {st['fallback']:>8,} global fallback")
print(f"  {'soil_weather':25s}: {'N/A':>10}  (no Crop column)")

print()
print("Columns sourced from each dataset:")
print("  crop_yield (backbone):  Region, Soil_Type, Crop, Rainfall_mm, Temperature_Celsius,")
print("                          Fertilizer_Used, Irrigation_Used, Weather_Condition,")
print("                          Days_to_Harvest, Yield_tons_per_hectare")
print("  smart_farming:          soil_pH, humidity_pct, sunlight_hours, irrigation_type")
print("  crop_recommendation:    N, P, K, temperature, humidity, ph, rainfall")

In [ ]:
# ====================================================================
# 8e.  FINAL COLUMN LIST
# ====================================================================

print("Final columns in agriculture_data.csv:")
for i, col in enumerate(enriched.columns, 1):
    dtype = enriched[col].dtype
    print(f"  {i:>2d}. {col:35s}  ({dtype})")

In [ ]:
# ====================================================================
# 8f.  HEAD(10) & SHAPE (user confirmation)
# ====================================================================

print("First 10 rows of the master dataset:")
display(enriched.head(10))
print(f"\nFinal shape: {enriched.shape[0]:,} rows x {enriched.shape[1]} columns")
print()
print("This data was built from YOUR provided files:")
print("  - data/crop_yield.csv (backbone, 1M rows)")
print("  - data/smart_farming.csv (per-crop aggregates via crop_type -> Crop)")
print("  - data/crop_recommendation.csv (per-crop aggregates via label -> Crop)")
print("  - data/soil_weather.csv (no Crop column -- global stats only)")

---
**Prompt Block 1 Complete.** The master dataset `agriculture_data.csv` is built from the real source files.